# Step 8 - Mask-Based Anomaly Segmentation Baselines with EoMT

This notebook is a copy of "**Segmentation Project-STEP_8_COCO+Cityscapes.ipynb**". \
To handle RAM and computational time limitations we decided to evaluate the **Fine-Tuned** model separately in this notebook, even though the logic implemented is the same and could potentially all the 3 checkpoints in ine single loop.

## 8.1 Environment Setup

In [1]:
import os
import sys
from google.colab import drive


print("Starting environment setup...")

# 1. Mount Google Drive to access project files and datasets
drive.mount('/content/drive')

# 2. Define and navigate to the EoMT directory within the project structure
project_folder = "/content/drive/MyDrive/FAIMDL Project"
repo_name = "MaskArchitectureAnomaly_CourseProject"
repo_path = os.path.join(project_folder, repo_name)
eval_folder = os.path.join(repo_path, "eval")
eomt_folder = os.path.join(repo_path, "eomt")

if os.path.exists(eomt_folder):
    %cd "{eomt_folder}"
    print(f"Directoty successfully changed to: {eomt_folder}")
else:
    print("ERROR: EoMT folder not found. Please check your project path.")

# 3. We update Python Paths to ensure all modules can be imported
paths_to_add = [repo_path, eval_folder, eomt_folder]
for p in paths_to_add:
    if p not in sys.path:
        sys.path.insert(0, p)


# 4. Clean Installation
# Install lightning which is required by EoMT
!pip install lightning -q

# Install ood-metrics without modifying existing dependencies (force NumPy 2.x)
!pip install --no-deps ood-metrics -q

# Install requirements but we exclude torch and torchvision (pre-installed on Colab) to protect the CUDA environment
!grep -v -E "torch==|torchvision==" requirements.txt > safe_reqs.txt
!pip install -r safe_reqs.txt -q

print("EoMT Environment prepared successfully!")

Starting environment setup...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/FAIMDL Project/MaskArchitectureAnomaly_CourseProject/eomt
Directoty successfully changed to: /content/drive/MyDrive/FAIMDL Project/MaskArchitectureAnomaly_CourseProject/eomt
EoMT Environment prepared successfully!


## 8.2 Configuration & Imports
In this section, we import the necessary libraries for Out-of-Distribution (OoD) evaluation, set seeds to ensure reproducibility, and configure the directory paths for the three checkpoints (COCO, Cityscapes, and Fine-Tuned) and the anomaly validation datasets.

In [2]:
import glob
import torch
import random
import numpy as np
import gc
from PIL import Image
from tqdm import tqdm
import yaml
import importlib
import warnings

from torchvision.transforms import Compose, Resize
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
from sklearn.metrics import average_precision_score
from ood_metrics import fpr_at_95_tpr
from lightning import seed_everything

In [3]:
seed_everything(0, verbose=False)

DEVICE_ID = 0
IGNORE_INDEX = 255

# Paths
PROJECT_FOLDER = "/content/drive/MyDrive/FAIMDL Project"
REPO_NAME = "MaskArchitectureAnomaly_CourseProject"
REPO_PATH = os.path.join(PROJECT_FOLDER, REPO_NAME)
EOMT_FOLDER = os.path.join(REPO_PATH, "eomt")

# Change working directory
os.chdir(EOMT_FOLDER)
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, "eval"))
sys.path.insert(0, EOMT_FOLDER)

# Paths for checkpoints
CHECKPOINTS = {
    "coco": os.path.join(PROJECT_FOLDER, "eomt_coco.bin"),
    "cityscapes": os.path.join(PROJECT_FOLDER, "eomt_cityscapes.bin"),
    "finetuned": os.path.join(PROJECT_FOLDER, "eomt_coco_finetuned.bin") #FIXME/debug correct file? Need to change file name?
}

# Paths for YAML configs
CONFIG_PATHS = {
    "coco": "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml",
    "cityscapes": "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
}

# Paths for OOD datasets
ANOMALY_DATASETS = {
    "FS_LostFound_full": os.path.join(PROJECT_FOLDER, "Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full"),
    "RoadAnomaly": os.path.join(PROJECT_FOLDER, "Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly"),
    "RoadAnomaly21": os.path.join(PROJECT_FOLDER, "Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21"),
    "fs_static": os.path.join(PROJECT_FOLDER, "Anomaly_Validation_Datasets/Validation_Dataset/fs_static"),
    "Road Obstacle": os.path.join(PROJECT_FOLDER, "Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21"),
}

# Configurations for each checkpoint
CHECKPOINT_CONFIG = {
    "coco": {
        "config": "coco",
        "img_size": (640, 640),
        "num_classes": 133,
        "num_queries": 200,
        "need_translation": True
    },
    "cityscapes": {
        "config": "cityscapes",
        "img_size": (1024, 1024),
        "num_classes": 19,
        "num_queries": 100,
        "need_translation": False
    },
    "finetuned": {
        "config": "cityscapes",
        "img_size": (640, 640),
        "num_classes": 19,
        "num_queries": 100, # We keep the numbers of query from Cityscapes
        "need_translation": False
    }

}

# COCO to Cityscapes mapping (same developed for step 4)
COCO_TO_CITYSCAPES = {
    # Things (instances: 0-79) - COCO to Cityscapes
    0: 11,      # person -> person
    1: 18,      # bicycle -> bicycle
    2: 13,      # car -> car
    3: 17,      # motorcycle -> motorcycle
    5: 15,      # bus -> bus
    6: 16,      # train -> train
    7: 14,      # truck -> truck
    9: 6,       # traffic light -> traffic light
    11: 7,      # stop sign -> traffic sign

    # Stuff (regions: 80-132) - COCO to Cityscapes
    82: 2,      # bridge -> building
    88: 8,      # flower -> vegetation
    90: 9,      # gravel -> terrain
    91: 2,      # house -> building
    97: 9,      # playingfield -> terrain
    100: 0,     # road -> road
    101: 2,     # roof -> building
    102: 9,     # sand -> terrain
    105: 9,     # snow -> terrain
    109: 3,     # wall-brick -> wall
    110: 3,     # wall-stone -> wall
    111: 3,     # wall-tile -> wall
    112: 3,     # wall-wood -> wall
    116: 8,     # tree-merged -> vegetation
    117: 4,     # fence-merged -> fence
    119: 10,    # sky-other-merged -> sky
    123: 1,     # pavement-merged -> sidewalk
    124: 9,     # mountain-merged -> terrain
    125: 8,     # grass-merged -> vegetation
    126: 9,     # dirt-merged -> terrain
    129: 2,     # building-other-merged -> building
    130: 9,     # rock-merged -> terrain
    131: 3,     # wall-other-merged -> wall
}

# Temperature Scaling parameters
TEMPERATURES = [0.1, 0.5, 0.75, 1.0, 1.1, 1.5, 2.0, 5.0, 10.0, 100.0]

# Transform
input_transform = Compose([
    Resize((512, 1024), Image.BILINEAR),
])

target_transform = Compose([
    Resize((512, 1024), Image.NEAREST),
])

print("Configuration loaded successfully!")

Configuration loaded successfully!


## 8.3 Usefull Functions
Here we define some usefull functions to be used later. These functions are helpful for loading the model architectures and extracting pixel-level predictions from the transformer's mask predictions.
These functions are inspired by/adapted from code in inference.ipynb

In [4]:
def load_model(checkpoint_name):
    """Loads the specified model checkpoint."""

    print(f"\t Loading {checkpoint_name} model...")

    cfg_name = CHECKPOINT_CONFIG[checkpoint_name]["config"]
    config_path = CONFIG_PATHS[cfg_name]

    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    img_size = CHECKPOINT_CONFIG[checkpoint_name]["img_size"]
    num_classes = CHECKPOINT_CONFIG[checkpoint_name]["num_classes"]
    num_queries = CHECKPOINT_CONFIG[checkpoint_name]["num_queries"]

    warnings.filterwarnings("ignore", message=r".*Attribute 'network' is an instance.*")
    warnings.filterwarnings("ignore", category=UserWarning, module=r"huggingface_hub\.utils\._auth")

    # Load encoder
    encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
    encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
    encoder = encoder_cls(img_size=img_size, **encoder_cfg.get("init_args", {}))

    # Load network
    network_cfg = config["model"]["init_args"]["network"]
    network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
    network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
    network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
    network_kwargs["num_q"] = num_queries
    network = network_cls(
        masked_attn_enabled=False,
        num_classes=num_classes,
        encoder=encoder,
        **network_kwargs,
    )

    # Load Lightning module
    lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
    model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}

    if "stuff_classes" in config["data"].get("init_args", {}):
        model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

    # We remove 'num_classes' from model_kwargs_final to avoid passing it twice to the model
    # The correct value comes from the checkpoint (num_classes variable), while model_kwargs_final may contain a value from the YAML config.
    model_kwargs_final = {k: v for k, v in model_kwargs.items()}
    if "num_classes" in model_kwargs_final:
        del model_kwargs_final["num_classes"]

    model = (
        lit_cls(
            img_size=img_size,
            num_classes=num_classes,
            network=network,
            **model_kwargs_final,
        )
        .eval()
        .to(DEVICE_ID)
    )

    # Load weights
    state_dict = torch.load(CHECKPOINTS[checkpoint_name], map_location=f"cuda:{DEVICE_ID}", weights_only=True)
    model.load_state_dict(state_dict, strict=False)

    print(f"\t Model loaded: {num_classes} classes, {num_queries} queries")

    return model

In [5]:
def extract_pixel_logits_eomt(model, img_tensor, img_size):
    """Extracts logits from the EoMT model."""
    model.eval()
    model.window_size = img_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img_tensor.to(DEVICE_ID)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], img_size, mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)

    return logits[0].cpu()  # [C, H, W]

In [6]:
def map_preds(pred_array):
    """Maps COCO predictions to Cityscapes labels."""
    mapped = np.full(pred_array.shape, IGNORE_INDEX)

    for coco_id, city_id in COCO_TO_CITYSCAPES.items():
        mapped[pred_array == coco_id] = city_id

    return mapped

## 8.4 Post-Hoc Anomaly Scoring Methods
In this section, we implement the mathematical formulations for the four post-hoc anomaly scoring methods:
* **MSP** (Maximum Softmax Probability)
* **MaxLogit**
* **Max Entropy**
* **RbA** (Rejected by All)

In [7]:
def compute_anomaly_scores(logits):
    """Computes the 4 anomaly scores: MSP, MaxLogit, MaxEntropy, RbA."""

    # 1. MaxLogit
    # We extract the highest logit for each pixel
    max_logits_raw = torch.max(logits, dim=0)[0].numpy()
    # Anomaly score: the LOWER the maximum logit, the more anomalous it is.
    # We use the negative of the raw logit.
    max_logit = -max_logits_raw

    # Normalize logits for numerical stability (for Softmax and Entropy only)
    logits_norm = logits - logits.max(dim=0, keepdim=True)[0]

    # 2. Max Softmax Probability (MSP)
    probs = torch.nn.functional.softmax(logits_norm, dim=0)
    max_probs = torch.max(probs, dim=0)[0].numpy()
    # Anomaly score: 1 - maximum confidence
    msp = 1.0 - max_probs

    # 3. Max Entropy
    epsilon = 1e-7
    # Entropy is already a metric of uncertainty (high entropy = high anomaly)
    entropy = -torch.sum(probs * torch.log(probs + epsilon), dim=0).numpy()

    # 4. RbA (Region-based Anomaly)
    # Following the definition of get_RbA() in evaluate_ood.py
    # For In-Distribution (ID) tanh values are high, so to get an anomaly score (OOD) we take the opposite:
    rba = -logits.tanh().sum(dim=0).numpy()

    return {
        "msp": msp,
        "maxlogit": max_logit,
        "maxentropy": entropy,
        "rba": rba
    }

In [8]:
def calculate_evalAnomaly_metrics(flat_gts, flat_scores):
    """Replicates the masking logic and metrics calculation from evalAnomaly.py."""
    # Removed because redundant with operations in process_dataset() and we have RAM issues
    # Moreover fpr_at_95_tpr does not require the arrays to be "ordered"
    '''
    ood_mask = (flat_gts == 1)
    ind_mask = (flat_gts == 0)

    ood_out = flat_scores[ood_mask]
    ind_out = flat_scores[ind_mask]

    ood_label = np.ones(len(ood_out))
    ind_label = np.zeros(len(ind_out))

    val_out = np.concatenate((ind_out, ood_out))
    val_label = np.concatenate((ind_label, ood_label))'''

    try:
        auprc = average_precision_score(flat_gts, flat_scores) * 100.0
        fpr = fpr_at_95_tpr(flat_scores, flat_gts) * 100.0
    except Exception:
        auprc, fpr = 0.0, 100.0

    return auprc, fpr

## 8.5 Main OOD Evaluation Loop
We infer for all the 3 checkpoints using the 5 validation datasets. For each image, we extract pixel-level logits, compute the anomaly scores and concatenate the valid pixels to calculate the global metrics: AuPRC and FPR95.

In [9]:
def process_dataset(model, dataset_path, img_size, dataset_name, return_raw_logits=False):
    img_dir = os.path.join(dataset_path, "images")
    if not os.path.exists(img_dir): return (None, None, None) if return_raw_logits else (None, None)

    img_files = [f for f in glob.glob(os.path.join(img_dir, "*.*")) if f.endswith(('.png', '.jpg', '.webp'))]
    if not img_files: return (None, None, None) if return_raw_logits else (None, None)
    print(f"\t Found {len(img_files)} images")

    all_scores_dict = {"msp": [], "maxlogit": [], "maxentropy": [], "rba": []}
    all_gts = []
    all_logits = []

    '''i = 0 #debug'''

    for img_path in tqdm(img_files, desc=f"\t Processing {dataset_name}", leave=False):
        try:
            # Load and prepare image
            img_pil = input_transform(Image.open(img_path).convert('RGB'))
            img_tensor = torch.from_numpy(np.array(img_pil)).permute(2, 0, 1)

            # Extract logits
            logits = extract_pixel_logits_eomt(model, img_tensor, img_size)

            # Load mask
            mask_path = img_path.replace("images", "labels_masks").rsplit(".", 1)[0] + ".png"

            if not os.path.exists(mask_path):
                continue

            ood_gts = np.array(target_transform(Image.open(mask_path)))

            # As in evalAnomaly.py
            if "RoadAnomaly" in mask_path:
                ood_gts = np.where((ood_gts == 2), 1, ood_gts)
            '''if "LostAndFound" in mask_path or "FS_LostFound" in mask_path:
                ood_gts = np.where((ood_gts == 0), 255, ood_gts)
                ood_gts = np.where((ood_gts == 1), 0, ood_gts)
                ood_gts = np.where((ood_gts > 1) & (ood_gts < 201), 1, ood_gts)'''

            if "Streethazard" in mask_path:
                ood_gts = np.where((ood_gts == 14), 255, ood_gts)
                ood_gts = np.where((ood_gts < 20), 0, ood_gts)
                ood_gts = np.where((ood_gts == 255), 1, ood_gts)

            if 1 not in np.unique(ood_gts):
                continue

            # We apply valid mask to discard IGNORE_INDEX
            valid_mask = (ood_gts == 0) | (ood_gts == 1)

            # We save logits if to be returned
            if return_raw_logits:
                if torch.is_tensor(logits):
                    valid_logits = logits[:, valid_mask].detach().cpu().half().numpy()
                else:
                    valid_logits = logits[:, valid_mask].astype(np.float16)
                all_logits.append(valid_logits)

            scores = compute_anomaly_scores(logits)

            for metric, score in scores.items():
                all_scores_dict[metric].append(score[valid_mask])
            all_gts.append(ood_gts[valid_mask])

            del logits, img_tensor, ood_gts, scores
            gc.collect()

            '''i += 1 #debug
            if i == 3: #debug
                break #debug'''

        except Exception:
            continue

    if not all_gts:
        return (None, None, None) if return_raw_logits else (None, None)

    # Force uint8 dtype to limit RAM use
    #flat_gts = np.concatenate(all_gts).astype(np.uint8)
    #del all_gts

    if return_raw_logits:
        # Force float32 dtype to limit RAM use
        #flat_logits = np.concatenate(all_logits, axis=1).astype(np.float32)
        #del all_logits
        return all_scores_dict, np.concatenate(all_gts).astype(np.uint8), np.concatenate(all_logits, axis=1).astype(np.float16)

    return all_scores_dict, np.concatenate(all_gts).astype(np.uint8)

In [10]:
print("OOD EVALUATION - ALL 3 CHECKPOINTS")

# Dictionary to collect final results
all_results = {}
# Dictionary to store results of temperature scaling (displayed in the next section)
temperature_scaling_results = {}

for checkpoint_name in ["finetuned"]:
    print(f"\n  CHECKPOINT: {checkpoint_name.upper()}\n")

    model = load_model(checkpoint_name)
    img_size = CHECKPOINT_CONFIG[checkpoint_name]["img_size"]
    need_translation = CHECKPOINT_CONFIG[checkpoint_name]["need_translation"]

    checkpoint_results = {}

    for dataset_name, dataset_path in ANOMALY_DATASETS.items():
        print(f"\n\t Dataset: {dataset_name}")

        is_finetuned = (checkpoint_name == "finetuned")

        # We want also the logits for the finetuned to later use them for temperature scaling
        if is_finetuned:
            scores_dict, flat_gts, flat_logits = process_dataset(
                model, dataset_path, img_size, dataset_name, return_raw_logits=True
            )
        else:
            scores_dict, flat_gts = process_dataset(
                model, dataset_path, img_size, dataset_name, return_raw_logits=False
            )

        if flat_gts is not None:
            dataset_results = {}
            print(f"\n\tResults for {dataset_name}:\n\t {'-'*50}")

            # 1. Standard evaluation for all the models
            for metric, scores_list in scores_dict.items():
                if len(scores_list) > 0:
                    flat_scores = np.concatenate(scores_list)

                    auprc, fpr95 = calculate_evalAnomaly_metrics(flat_gts, flat_scores)
                    dataset_results[metric] = {"auprc": auprc, "fpr95": fpr95}

                    print(f"\t {metric.upper():12} | AUPRC: {auprc:6.2f}%  | FPR95: {fpr95:6.2f}%")

            checkpoint_results[dataset_name] = dataset_results

            # 2. TEMPERATURE SCALING evaluation
            if is_finetuned:
                print(f"\n\t Performing Temperature Scaling and computing metrics...\n")

                # Initialise the storage for this dataset
                temperature_scaling_results[dataset_name] = {
                    "metrics": {},
                    "best": {
                        "msp": {"T": 1.0, "auprc": 0, "fpr95": 100},
                        "maxentropy": {"T": 1.0, "auprc": 0, "fpr95": 100}
                    }
                }

                best_temp = temperature_scaling_results[dataset_name]["best"]

                logits_tensor = torch.as_tensor(flat_logits, dtype=torch.float16)

                num_pixels = logits_tensor.shape[1]
                CHUNK_SIZE = 10000000

                for T in TEMPERATURES:
                    msp_chunks = []
                    entropy_chunks = []

                    # We process logits in chunks to avoid RAM issues
                    for start_idx in range(0, num_pixels, CHUNK_SIZE):
                        end_idx = min(start_idx + CHUNK_SIZE, num_pixels)

                        # Extract chunk and convert to float32 to allow safe scores computation
                        chunk_logits = logits_tensor[:, start_idx:end_idx].float()

                        scaled_chunk = chunk_logits / T
                        scores_chunk = compute_anomaly_scores(scaled_chunk)

                        msp_chunks.append(scores_chunk["msp"])
                        entropy_chunks.append(scores_chunk["maxentropy"])

                        # Clean memory for this chunk
                        del chunk_logits, scaled_chunk, scores_chunk

                    # Concatenate arrays
                    flat_msp = np.concatenate(msp_chunks)
                    flat_entropy = np.concatenate(entropy_chunks)

                    auprc_msp, fpr_msp = calculate_evalAnomaly_metrics(flat_gts, flat_msp)
                    auprc_entropy, fpr_entropy = calculate_evalAnomaly_metrics(flat_gts, flat_entropy)

                    # Save the metrics
                    temperature_scaling_results[dataset_name]["metrics"][T] = {
                        "msp": {"auprc": auprc_msp, "fpr95": fpr_msp},
                        "maxentropy": {"auprc": auprc_entropy, "fpr95": fpr_entropy}
                    }

                    # Save the best settings
                    if auprc_msp > best_temp["msp"]["auprc"]:
                        best_temp["msp"] = {"T": T, "auprc": auprc_msp, "fpr95": fpr_msp}
                    if auprc_entropy > best_temp["maxentropy"]["auprc"]:
                        best_temp["maxentropy"] = {"T": T, "auprc": auprc_entropy, "fpr95": fpr_entropy}

                    # Clean the RAM (at each T iteration)
                    del flat_msp, flat_entropy
                    torch.cuda.empty_cache()

                # Clean the RAM (logits can take much space)
                del flat_logits, logits_tensor
                torch.cuda.empty_cache()

            # Clean the RAM
            del flat_gts
            torch.cuda.empty_cache()
            gc.collect()

        else:
            print(f"\t No valid data processed")

    all_results[checkpoint_name] = checkpoint_results
    del model
    torch.cuda.empty_cache()
    gc.collect()

OOD EVALUATION - ALL 3 CHECKPOINTS

  CHECKPOINT: FINETUNED

	 Loading finetuned model...


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

	 Model loaded: 19 classes, 100 queries

	 Dataset: FS_LostFound_full
	 Found 100 images



	Results for FS_LostFound_full:
	 --------------------------------------------------
	 MSP          | AUPRC:  44.68%  | FPR95:  38.19%
	 MAXLOGIT     | AUPRC:  44.66%  | FPR95:  41.60%
	 MAXENTROPY   | AUPRC:  44.47%  | FPR95:  38.06%
	 RBA          | AUPRC:  44.37%  | FPR95:  91.54%

	 Performing Temperature Scaling and computing metrics...


	 Dataset: RoadAnomaly
	 Found 60 images



	Results for RoadAnomaly:
	 --------------------------------------------------
	 MSP          | AUPRC:  73.00%  | FPR95:  19.81%
	 MAXLOGIT     | AUPRC:  72.92%  | FPR95:  19.23%
	 MAXENTROPY   | AUPRC:  74.30%  | FPR95:  18.64%
	 RBA          | AUPRC:  71.93%  | FPR95:  17.17%

	 Performing Temperature Scaling and computing metrics...


	 Dataset: RoadAnomaly21
	 Found 10 images



	Results for RoadAnomaly21:
	 --------------------------------------------------
	 MSP          | AUPRC:  91.46%  | FPR95:   3.84%
	 MAXLOGIT     | AUPRC:  91.25%  | FPR95:   3.78%
	 MAXENTROPY   | AUPRC:  91.25%  | FPR95:   3.62%
	 RBA          | AUPRC:  86.95%  | FPR95:  25.49%

	 Performing Temperature Scaling and computing metrics...


	 Dataset: fs_static
	 Found 30 images



	Results for fs_static:
	 --------------------------------------------------
	 MSP          | AUPRC:  79.38%  | FPR95:  10.11%
	 MAXLOGIT     | AUPRC:  79.74%  | FPR95:   9.66%
	 MAXENTROPY   | AUPRC:  80.27%  | FPR95:   9.31%
	 RBA          | AUPRC:  82.49%  | FPR95:   4.82%

	 Performing Temperature Scaling and computing metrics...


	 Dataset: Road Obstacle
	 Found 30 images



	Results for Road Obstacle:
	 --------------------------------------------------
	 MSP          | AUPRC:  79.53%  | FPR95:   2.62%
	 MAXLOGIT     | AUPRC:  79.49%  | FPR95:   2.60%
	 MAXENTROPY   | AUPRC:  79.86%  | FPR95:   2.61%
	 RBA          | AUPRC:  76.97%  | FPR95:   4.78%

	 Performing Temperature Scaling and computing metrics...



## 8.6 Temperature Scaling (Post-Hoc Calibration)
Using pre-extracted logits from the Fine-Tuned model, we try a set of values for the temperature $T$ parameter, trying to calibrate the Softmax probabilities. The goal is to find the ideal $T$ to maximize AuPRC and minimize FPR95 for MSP method.\
The actual computation is performed inside the `process_dataset()` function to make the computation more efficient.\
The following cell shows the results of this analysis.

In [11]:

print(f"TEMPERATURE SCALING - FINE-TUNED CHECKPOINT\n")

if not temperature_scaling_results:
    print("No data found on Temperature Scaling. Run the previous cell fist.")
else:
    for dataset_name, data in temperature_scaling_results.items():
        print(f"\n\tTemperature Scaling on: {dataset_name}\n\t " + "-" * 77)
        print(f"\t {'Temperature':<12} {'MSP AUPRC':^15} {'MSP FPR95':^15} {'Entropy AUPRC':^15} {'Entropy FPR95':^15}\n\t {'-'*77}")

        metrics_by_temp = data["metrics"]
        best_temp = data["best"]

        for T in sorted(metrics_by_temp.keys()):
            msp = metrics_by_temp[T]["msp"]
            entropy = metrics_by_temp[T]["maxentropy"]

            print(f"\t T={T:<10.2f}|{msp['auprc']:^15.2f}|{msp['fpr95']:^15.2f}|{entropy['auprc']:^15.2f}|{entropy['fpr95']:^15.2f}")

        print(f"  {'-'*87}")
        print(f"   Best MSP:     T = {best_temp['msp']['T']:.2f}  (AUPRC: {best_temp['msp']['auprc']:.2f}%, FPR95: {best_temp['msp']['fpr95']:.2f}%)")
        print(f"   Best Entropy: T = {best_temp['maxentropy']['T']:.2f}  (AUPRC: {best_temp['maxentropy']['auprc']:.2f}%, FPR95: {best_temp['maxentropy']['fpr95']:.2f}%)")


TEMPERATURE SCALING - FINE-TUNED CHECKPOINT


	Temperature Scaling on: FS_LostFound_full
	 -----------------------------------------------------------------------------
	 Temperature     MSP AUPRC       MSP FPR95     Entropy AUPRC   Entropy FPR95 
	 -----------------------------------------------------------------------------
	 T=0.10      |     44.22     |     38.27     |     44.76     |     38.08     
	 T=0.50      |     44.65     |     38.07     |     45.20     |     37.88     
	 T=0.75      |     44.67     |     38.08     |     44.87     |     37.83     
	 T=1.00      |     44.68     |     38.07     |     44.41     |     38.00     
	 T=1.10      |     44.68     |     38.07     |     44.37     |     38.07     
	 T=1.50      |     44.68     |     38.08     |     43.80     |     38.40     
	 T=2.00      |     44.69     |     38.18     |     43.17     |     38.77     
	 T=5.00      |     44.70     |     38.22     |     40.20     |     39.00     
	 T=10.00     |     44.72     |     38.1